In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
RAW_PATH = "/Volumes/projeto_spark/raw/raw_files"

print(RAW_PATH)

In [0]:
df_orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW_PATH}/orders.csv")
)

display(df_orders)

In [0]:
df_orders.printSchema()

In [0]:
print(f"Quantidade de registros: {df_orders.count():,}")

In [0]:
display(
    df_orders.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_orders.columns
    ])
)

In [0]:
display(df_orders.describe())

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS projeto_spark.bronze
""")

In [0]:
arquivos = [
    "categories",
    "customers",
    "employees",
    "order_items",
    "orders",
    "payments",
    "products",
    "promotions",
    "returns",
    "shipments",
    "stores",
    "suppliers"
]

for arquivo in arquivos:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_PATH}/{arquivo}.csv")
    )
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"projeto_spark.bronze.{arquivo}")
    )
    
    print(f"{arquivo} carregado")

In [0]:
contagens_bronze = []

for tabela in arquivos:
    quantidade = spark.table(
        f"projeto_spark.bronze.{tabela}"
    ).count()

    contagens_bronze.append((tabela, quantidade))

for tabela, quantidade in contagens_bronze:
    print(f"{tabela:15} → {quantidade:,} registros")

total = sum(qtd for _, qtd in contagens_bronze)

print("-" * 45)
print(f"TOTAL DE REGISTROS → {total:,}")

#Silver

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS projeto_spark.silver
""")

In [0]:
tabelas_silver = [
    "categories",
    "customers",
    "products",
    "orders",
    "order_items"
]

for tabela in tabelas_silver:
    df = (
        spark.table(f"projeto_spark.bronze.{tabela}")
    )
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"projeto_spark.silver.{tabela}")
    )

    print(f"{tabela} criada na Silver")

In [0]:
orders_silver = spark.table("projeto_spark.silver.orders")
order_items_silver = spark.table("projeto_spark.silver.order_items")

df_vendas = (
    orders_silver.alias("o")
    .join(
        order_items_silver.alias("oi"),
        F.col("o.order_id") == F.col("oi.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("o.store_id"),
        F.col("o.order_date"),
        F.col("o.promotion_id"),
        F.col("oi.order_item_id"),
        F.col("oi.product_id"),
        F.col("oi.qty"),
        F.col("oi.price").alias("item_price")
    )
)

print(f"JOIN orders + order_items realizado")
print(f"Quantidade de registros: {df_vendas.count()}")

display(df_vendas.limit(10))

In [0]:
products_silver = spark.table("projeto_spark.silver.products")

df_vendas_produtos = (
    df_vendas.alias("v")
    .join(
        products_silver.alias("p"),
        F.col("v.product_id") == F.col("p.product_id"),
        "inner"
    )
    .select(
        F.col("v.order_id"),
        F.col("v.customer_id"),
        F.col("v.store_id"),
        F.col("v.order_date"),
        F.col("v.promotion_id"),
        F.col("v.order_item_id"),
        F.col("v.product_id"),
        F.col("p.category_id"),
        F.col("p.supplier_id"),
        F.col("v.qty"),
        F.col("v.item_price")
    )
)

print("JOIN order_items + products realizado")
print(f"Quantidade de registros: {df_vendas_produtos.count()}")

display(df_vendas_produtos.limit(10))

In [0]:
categories_silver = spark.table("projeto_spark.silver.categories")
df_vendas_categorias = (
    df_vendas_produtos.alias("v")
    .join(

        categories_silver.alias("c"),

        F.col("v.category_id") == F.col("c.category_id"),

        "inner"

    )
    .select(

        F.col("v.order_id"),
        F.col("v.customer_id"),
        F.col("v.store_id"),
        F.col("v.order_date"),
        F.col("v.promotion_id"),
        F.col("v.order_item_id"),
        F.col("v.product_id"),
        F.col("v.category_id"),
        F.col("c.category_name"),
        F.col("v.supplier_id"),
        F.col("v.qty"),
        F.col("v.item_price")
    )
)

print("JOIN products + categories realizado")

print(f"Quantidade de registros: {df_vendas_categorias.count()}")

display(df_vendas_categorias.limit(10))

In [0]:
customers_silver = spark.table("projeto_spark.silver.customers")
df_vendas_final = (
    df_vendas_categorias.alias("v")
    .join(

        customers_silver.alias("c"),

        F.col("v.customer_id") == F.col("c.customer_id"),

        "inner"

    )
    .select(

        F.col("v.order_id"),
        F.col("v.customer_id"),
        F.col("c.city"),
        F.col("c.signup_date"),
        F.col("v.store_id"),
        F.col("v.order_date"),
        F.col("v.promotion_id"),
        F.col("v.order_item_id"),
        F.col("v.product_id"),
        F.col("v.category_id"),
        F.col("v.category_name"),
        F.col("v.supplier_id"),
        F.col("v.qty"),
        F.col("v.item_price")

    )
)

print("JOIN vendas + customers realizado")

print(f"Quantidade de registros: {df_vendas_final.count()}")

display(df_vendas_final.limit(10))

In [0]:
df_vendas_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projeto_spark.silver.vendas")

print("Tabela projeto_spark.silver.vendas criada com sucesso")

In [0]:
df_silver_vendas = spark.table("projeto_spark.silver.vendas")

print(f"Quantidade de registros: {df_silver_vendas.count()}")

display(df_silver_vendas.limit(10))

In [0]:
df_explain_join = (

    spark.table("projeto_spark.silver.orders").alias("o")

    .join(
        spark.table("projeto_spark.silver.order_items").alias("oi"),
        F.col("o.order_id") == F.col("oi.order_id"),
        "inner"
    )

    .join(
        spark.table("projeto_spark.silver.products").alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "inner"
    )

    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("oi.product_id"),
        F.col("oi.qty"),
        F.col("p.category_id"),
        F.col("p.supplier_id")
    )
)

df_explain_join.explain("formatted")

In [0]:
df_explain_broadcast = (

    spark.table("projeto_spark.silver.orders").alias("o")

    .join(
        spark.table("projeto_spark.silver.order_items").alias("oi"),
        F.col("o.order_id") == F.col("oi.order_id"),
        "inner"
    )

    .join(
        F.broadcast(
            spark.table("projeto_spark.silver.products").alias("p")
        ),
        F.col("oi.product_id") == F.col("p.product_id"),
        "inner"
    )

    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("oi.product_id"),
        F.col("oi.qty"),
        F.col("p.category_id"),
        F.col("p.supplier_id")
    )
)

df_explain_broadcast.explain("formatted")

#Gold

In [0]:
df_gold_categoria = (
    spark.table("projeto_spark.silver.vendas")
    .withColumn(
        "valor_item",
        F.col("qty") * F.col("item_price")
    )
    .groupBy(
        "category_id",
        "category_name"
    )
    .agg(
        F.sum("qty").alias("quantidade_vendida"),
        F.sum("valor_item").alias("valor_total")
    )
    .orderBy(
        F.desc("valor_total")
    )
)

display(df_gold_categoria)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS projeto_spark.gold
""")

In [0]:
df_gold_categoria.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projeto_spark.gold.vendas_por_categoria")

print("Tabela Gold criada: projeto_spark.gold.vendas_por_categoria")

In [0]:
df_gold_categoria_validacao = spark.table(
    "projeto_spark.gold.vendas_por_categoria"
)

print(
    f"Quantidade de categorias: "
    f"{df_gold_categoria_validacao.count()}"
)

display(df_gold_categoria_validacao)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

df_gold_ranking = (
    spark.table("projeto_spark.gold.vendas_por_categoria")
    .withColumn(
        "ranking_categoria",
        F.dense_rank().over(
            Window.orderBy(F.desc("valor_total"))
        )
    )
    .orderBy("ranking_categoria")
)

display(df_gold_ranking)

In [0]:
df_gold_pivot = (

    spark.table("projeto_spark.silver.vendas")

    .groupBy("city")

    .pivot("category_name")

    .agg(
        F.sum("qty")
    )
)

display(df_gold_pivot)

In [0]:
%sql
SELECT
    city,
    SUM(qty * item_price) AS valor_total
FROM projeto_spark.silver.vendas
GROUP BY city
ORDER BY valor_total DESC;

In [0]:
%sql
EXPLAIN FORMATTED
SELECT
    city,
    SUM(qty * item_price) AS valor_total
FROM projeto_spark.silver.vendas
GROUP BY city
ORDER BY valor_total DESC;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN projeto_spark;

In [0]:
%sql
SHOW TABLES IN projeto_spark.bronze;

In [0]:
%sql
SHOW TABLES IN projeto_spark.silver;

In [0]:
%sql
SHOW TABLES IN projeto_spark.gold;

In [0]:
%sql
DESCRIBE EXTENDED projeto_spark.silver.vendas;

In [0]:
%sql
SELECT current_user() AS usuario_atual;

In [0]:
%sql
GRANT SELECT
ON TABLE projeto_spark.gold.vendas_por_categoria
TO `usuario@exemplo.com`;

In [0]:
%sql
SHOW GRANTS ON TABLE projeto_spark.gold.vendas_por_categoria;

In [0]:
df_gold_final = spark.table("projeto_spark.gold.vendas_por_categoria")

print("Tabela Gold carregada")
print(f"Quantidade de registros: {df_gold_final.count()}")

df_gold_final.printSchema()

display(df_gold_final.limit(10))

In [0]:
print("VALIDAÇÃO FINAL DO PIPELINE")

bronze_orders = spark.table("projeto_spark.bronze.orders").count()
bronze_order_items = spark.table("projeto_spark.bronze.order_items").count()

silver_orders = spark.table("projeto_spark.silver.orders").count()
silver_order_items = spark.table("projeto_spark.silver.order_items").count()
silver_vendas = spark.table("projeto_spark.silver.vendas").count()

gold_categoria = spark.table("projeto_spark.gold.vendas_por_categoria").count()

print(f"Bronze orders:       {bronze_orders:,}")
print(f"Silver orders:       {silver_orders:,}")
print(f"Bronze order_items:  {bronze_order_items:,}")
print(f"Silver order_items:  {silver_order_items:,}")
print(f"Silver vendas:       {silver_vendas:,}")
print(f"Gold categorias:     {gold_categoria:,}")

print("\nRESULTADOS ")

print(
    "Orders preservados:"
    if bronze_orders == silver_orders
    else "Divergência em orders"
)

print(
    "Order_items preservados:"
    if bronze_order_items == silver_order_items
    else "Divergência em order_items"
)

print(
    "Silver vendas = 600.000:"
    if silver_vendas == 600000
    else "Quantidade inesperada na Silver vendas"
)

print(
    "Gold = 30 categorias:"
    if gold_categoria == 30
    else "Quantidade inesperada na Gold"
)

print("\nValidação final concluída")